# Session 16: Pandas practice (1)

In this session we will use Spotify data to practice.

These are the topics to be covered:

- Reading data from JSON files
- Concatenating dataframes
- Filtering dataframes
- Grouping and aggregating data

In [ ]:
import pandas as pd

## Exercise 1: Reading data from JSON files

Convert the two JSON files into a single dataframe.

In [ ]:
"""
select * from spotify_0
union all
select * from spotify_1
"""

In [14]:
spotify_0 = pd.read_json('/Users/dgh/Desktop/PDA/pda1/data/StreamingHistory_music_0.json')
spotify_1 = pd.read_json('/Users/dgh/Desktop/PDA/pda1/data/StreamingHistory_music_1.json')

spotify = pd.concat([spotify_0, spotify_1])

spotify.sample(5)

,endTime,artistName,trackName,msPlayed
1723,2024-07-31 12:38,Mujeres,Las Victorias y Derrotas,206026
6679,2024-11-17 11:19,Bahamas,Caught Me Thinkin,180200
344,2025-02-26 11:40,FloFilz,Doves,166069
8353,2024-12-17 17:25,Minami Deutsch,Steller Waffle,86506
5421,2024-10-26 07:48,Offthewally,Marmalade,124852


## Exercise 2: 

- Who's the artist with the most tracks in the dataset?
- Who's the artist I've listened to for the longest total time?
- What's the song that I've listened to the most times?
- What's the song I've played for the longest total time?

In [25]:
# - Who's the artist with the most tracks in the dataset?

spotify['artistName'].value_counts().idxmax()

'Mujeres'

In [21]:
# - Who's the artist I've listened to for the longest total time?

spotify.groupby('artistName')['msPlayed'].sum().idxmax()

'Mujeres'

In [62]:
# - What's the song that I've listened to the most times?

spotify['trackName'].value_counts().idxmax()

'Unknown Track'

In [29]:
# - What's the song I've played for the longest total time?

spotify.groupby('trackName')['msPlayed'].sum().idxmax()

'Un Gesto Brillante'

## Exercise 3:

How many artists whose songs (individually counted) I have listened to for 1 minute or less?

In [30]:
spotify.head()

,endTime,artistName,trackName,msPlayed
0,2024-06-11 06:27,Nacho Vegas,Lo Que Comen las Brujas,83586
1,2024-06-11 10:21,Nacho Vegas,Lo Que Comen las Brujas,114248
2,2024-06-11 10:23,bigott,She is My Man,14411
3,2024-06-11 10:23,interrogación amor,tú y yo,3482
4,2024-06-11 10:23,Doble Pletina,Cruzo los dedos,115450


In [38]:
# convert ms into minutes

spotify['minPlayed'] = spotify['msPlayed'] / 1000 / 60

spotify[spotify['minPlayed'] <= 1]#['artistName'].nunique()

,endTime,artistName,trackName,msPlayed,minPlayed
2,2024-06-11 10:23,bigott,She is My Man,14411,0.240183
3,2024-06-11 10:23,interrogación amor,tú y yo,3482,0.058033
12,2024-06-11 11:10,Los lagos de Hinault,El Verano No Nos Quiere,47206,0.786767
13,2024-06-11 11:10,Manos De Topo,Mentirosa,2739,0.045650
15,2024-06-11 11:14,Doble Pletina,Artista Revelación,4901,0.081683
...,...,...,...,...,...
4911,2025-06-11 16:26,KNEECAP,Get Your Brits Out,8255,0.137583
4918,2025-06-11 16:47,KNEECAP,H.O.O.D,14410,0.240167
4920,2025-06-11 16:48,KNEECAP,3CAG,9664,0.161067
4921,2025-06-11 16:50,KNEECAP,Interlude: Making Headlines,27175,0.452917


## Exercise 4: 

What's the artist with the longest song?

In [50]:
spotify.sort_values(by='msPlayed', ascending=False).head(1)['artistName'].values[0]

'Los Planetas'

In [49]:
spotify.sort_values(by='msPlayed', ascending=False).iloc[0]['artistName']

'Los Planetas'

In [54]:
spotify.set_index('artistName')['msPlayed'].idxmax()

'Los Planetas'

In [60]:
spotify.groupby('artistName')['msPlayed'].max().idxmax()

'Los Planetas'

## Interlude: datetime objects

In Pandas, we can convert strings to datetime objects using the `pd.to_datetime()` function. This is useful for filtering and manipulating date and time data.

Also, it allows us to extract specific components like year, month, day, hour, etc.

In [66]:
spotify['endTime'] = pd.to_datetime(spotify['endTime'])

spotify['endTime'].dtype

dtype('<M8[ns]')

In [67]:
spotify['endTime'] = pd.to_datetime(spotify['endTime'])

spotify['date'] = spotify['endTime'].dt.date
spotify['year'] = spotify['endTime'].dt.year
spotify['month'] = spotify['endTime'].dt.month
spotify['day'] = spotify['endTime'].dt.day
spotify['weekday'] = spotify['endTime'].dt.weekday
spotify['hour'] = spotify['endTime'].dt.hour
spotify['minute'] = spotify['endTime'].dt.minute
spotify['is_weekend'] = spotify['weekday'].isin([5, 6])

spotify.head()

,endTime,artistName,trackName,msPlayed,minPlayed,date,year,month,day,weekday,hour,minute,is_weekend
0,2024-06-11 06:27:00,Nacho Vegas,Lo Que Comen las Brujas,83586,1.393100,2024-06-11,2024,6,11,1,6,27,False
1,2024-06-11 10:21:00,Nacho Vegas,Lo Que Comen las Brujas,114248,1.904133,2024-06-11,2024,6,11,1,10,21,False
2,2024-06-11 10:23:00,bigott,She is My Man,14411,0.240183,2024-06-11,2024,6,11,1,10,23,False
3,2024-06-11 10:23:00,interrogación amor,tú y yo,3482,0.058033,2024-06-11,2024,6,11,1,10,23,False
4,2024-06-11 10:23:00,Doble Pletina,Cruzo los dedos,115450,1.924167,2024-06-11,2024,6,11,1,10,23,False


## Exercise 5:

Convert the `weekday` column from 0 to 6 to the actual names of the days of the week (e.g., 0 -> "Monday", 1 -> "Tuesday", etc.).

In [76]:
# not recommended

weekday_dict = dict(zip(
    [0, 1, 2, 3, 4, 5, 6],
    ['m', 't', 'w', 'th', 'f', 's', 'su']
))

spotify['weekday_name'] = [
    weekday_dict[x] for x in spotify['weekday']
]

spotify[['weekday', 'weekday_name']]

,weekday,weekday_name
0,1,t
1,1,t
2,1,t
3,1,t
4,1,t
...,...,...
4920,2,w
4921,2,w
4922,2,w
4923,2,w


In [74]:
weekday_dict = dict(zip(
    [0, 1, 2, 3, 4, 5, 6],
    ['m', 't', 'w', 'th', 'f', 's', 'su']
))

spotify['weekday_name'] = spotify['weekday'].map(weekday_dict)

spotify[['weekday', 'weekday_name']]

,weekday,weekday_name
0,1,t
1,1,t
2,1,t
3,1,t
4,1,t
...,...,...
4920,2,w
4921,2,w
4922,2,w
4923,2,w


In [72]:
weekday_list = ['m', 't', 'w', 'th', 'f', 's', 'su']

weekday_map = lambda x: weekday_list[x]

spotify['weekday_name'] = spotify['weekday'].map(weekday_map)

spotify[['weekday', 'weekday_name']]

,weekday,weekday_name
0,1,t
1,1,t
2,1,t
3,1,t
4,1,t
...,...,...
4920,2,w
4921,2,w
4922,2,w
4923,2,w


## Exercise 6:

What's the total time of music per month I've listened to?

In [77]:
spotify.groupby('month')['msPlayed'].sum()

month
1     109814357
2     176193585
3     190331456
4     189100140
5     241032433
6     150518804
7     189664614
8     109491974
9     207549762
10    210755437
11    240888073
12    174297283
Name: msPlayed, dtype: int64

## Exercise 7:

What's the average number of artists played per weekday?

In [90]:
unique_artists_per_weekday = spotify.groupby('weekday')['artistName'].nunique()

total_unique_artists = spotify['artistName'].nunique()

unique_artists_per_weekday

weekday
0    546
1    577
2    711
3    513
4    505
5    599
6    442
Name: artistName, dtype: int64

## Exercise 8:

What could you say of my music listening habits based on the hourly data?

In [96]:
spotify.describe()

,endTime,msPlayed,minPlayed,year,month,day,weekday,hour,minute
count,14925,1.492500e+04,14925.000000,14925.000000,14925.000000,14925.000000,14925.000000,14925.000000,14925.000000
mean,2024-12-14 15:33:24.434170880,1.467094e+05,2.445157,2024.410452,7.017152,16.086968,2.651591,12.629481,29.853467
min,2024-06-11 06:27:00,0.000000e+00,0.000000,2024.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,2024-09-19 09:26:00,5.474100e+04,0.912350,2024.000000,4.000000,9.000000,1.000000,10.000000,15.000000
50%,2024-11-29 10:22:00,1.632420e+05,2.720700,2024.000000,7.000000,16.000000,2.000000,12.000000,30.000000
75%,2025-03-18 07:52:00,2.111400e+05,3.519000,2025.000000,10.000000,24.000000,4.000000,15.000000,45.000000
max,2025-06-11 16:54:00,1.644085e+06,27.401417,2025.000000,12.000000,31.000000,6.000000,23.000000,59.000000
std,NaN,9.924951e+04,1.654159,0.491932,3.322878,8.741052,1.865920,3.747393,17.379741


In [95]:
spotify.groupby('hour')['minPlayed'].sum()

hour
0       74.476250
1       62.672100
2       60.343450
3       57.052867
4       88.008367
5      105.551483
6      511.655833
7     1341.417650
8     2685.221100
9     3639.713817
10    4201.478050
11    3456.002250
12    2542.895150
13    3331.487450
14    3195.072983
15    2828.457050
16    2870.753150
17    1929.763933
18    1487.868817
19     682.563767
20     371.080067
21     584.866800
22     263.144717
23     122.418200
Name: minPlayed, dtype: float64

## Exercise 9:

Extract all the distinct artists I have listened to between 0am and 7am

## Exercise 10:

What's the month/weekday/hour where I've listened to the most music?

## Exercise 11:

On May 10th 2025, I hosted a party. When did the last guest leave?

In [97]:
spotify

,endTime,artistName,trackName,msPlayed,minPlayed,date,year,month,day,weekday,hour,minute,is_weekend,weekday_name
0,2024-06-11 06:27:00,Nacho Vegas,Lo Que Comen las Brujas,83586,1.393100,2024-06-11,2024,6,11,1,6,27,False,t
1,2024-06-11 10:21:00,Nacho Vegas,Lo Que Comen las Brujas,114248,1.904133,2024-06-11,2024,6,11,1,10,21,False,t
2,2024-06-11 10:23:00,bigott,She is My Man,14411,0.240183,2024-06-11,2024,6,11,1,10,23,False,t
3,2024-06-11 10:23:00,interrogación amor,tú y yo,3482,0.058033,2024-06-11,2024,6,11,1,10,23,False,t
4,2024-06-11 10:23:00,Doble Pletina,Cruzo los dedos,115450,1.924167,2024-06-11,2024,6,11,1,10,23,False,t
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4920,2025-06-11 16:48:00,KNEECAP,3CAG,9664,0.161067,2025-06-11,2025,6,11,2,16,48,False,w
4921,2025-06-11 16:50:00,KNEECAP,Interlude: Making Headlines,27175,0.452917,2025-06-11,2025,6,11,2,16,50,False,w
4922,2025-06-11 16:50:00,KNEECAP,Fine Art,125184,2.086400,2025-06-11,2025,6,11,2,16,50,False,w
4923,2025-06-11 16:53:00,KNEECAP,I bhFiacha Linne,187316,3.121933,2025-06-11,2025,6,11,2,16,53,False,w


In [105]:
spotify[spotify['date'] == pd.to_datetime('2025-05-10')]

,endTime,artistName,trackName,msPlayed,minPlayed,date,year,month,day,weekday,hour,minute,is_weekend,weekday_name


In [103]:
spotify[
    pd.to_datetime(spotify['endTime'].dt.date == pd.to_datetime('2025-05-10').date)
]['endTime'].max()

TypeError: dtype bool cannot be converted to datetime64[ns]